# Teton Classifier Notebook

Random Forest Classifier for the Teton Range, based off of Yang et al. (2023)

## Imports and initializations

In [1]:
import os
import ee
import geemap
import pandas as pd


In [2]:
ee.Authenticate()

True

In [3]:
import os
import ee
import geemap
import pandas as pd

EE_PROJECT = "teton-classifier"
# Alpine-ish mask threshold for this first pass. Adjust after inspecting the map.
ALPINE_MIN_ELEV_M = 2200
YEAR = 2023
START_DATE = f"{YEAR}-06-15"
END_DATE = f"{YEAR}-09-20"
SCALE = 10
SEED = 42

if EE_PROJECT:
    ee.Initialize(project=EE_PROJECT)
else:
    ee.Initialize()
print("Earth Engine initialized.")

Earth Engine initialized.


## Map, draw study boundary or use preset box


In [4]:
Map = geemap.Map(center=[43.75, -110.75], zoom=10)
aoi = ee.Geometry.Rectangle([-111.05, 43.35, -110.45, 44.05])
Map.addLayer(aoi, {"color": "yellow"}, "AOI")
Map

Map(center=[43.75, -110.75], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprigh…

## Gather data

In [5]:
features = [
    # Sentinel-2 reflectance
    "B2", "B3", "B4",      # blue, green, red
    "B8",                 # near infrared
    "B11", "B12",         # shortwave infrared

    # Spectral indices
    "NDVI",               # vegetation greenness
    "NDWI",               # water / wetness
    "NDSI",               # snow / ice
    "BSI",                # bare soil / rock tendency

    # Terrain
    "elevation",
    "slope",
    "aspect"
]

In [ ]:
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60))
)